# cWave interactions with Gaslab

This notebook was translated from the original MATLAB live script. Examples that rely on plotting helpers or MATLAB-only convenience APIs not yet ported to Python are kept as notes or placeholders.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

from gaslab import GasState
from gaslab.relations import arstar, oblique


© Tim Colonius, California Institute of Technology

Revised: July 11, 2023

Regular reflection of shocks of the opposite family.  A cananocial case is shown in the sketch, and this configuration is possible when the angles \theta and \theta' are sufficiently small.  

For given values of \theta, \theta' and M_1, the solution, if it exists, is given by matching the flow direction and pressure between regions 3 and 3'.  

Example: Take M_1=3, \theta=15^0, \theta'=10^0.  Detemine M_3 and M_{3'} and their common flow angle and pressure (relative to the stagnation pressure in region 1).

Create functions that give the states in regions 3 and 3' as a function of the unknown flow angle there.  We use the wave family arguments and signed angles:

In [2]:
upper = lambda ang: GasState.init(3, 1.4).deflect(-10*np.pi/180, fam=-1).deflect(ang, fam=+1)
lower = lambda ang: GasState.init(3, 1.4).deflect(+15*np.pi/180, fam=+1).deflect(ang, fam=-1)


and we determine the angle to match their pressure

In [3]:
zerofun = lambda ang: upper(ang).pres - lower(ang).pres
ang = root_scalar(zerofun, bracket=[-20 * np.pi / 180, 20 * np.pi / 180], method="brentq").root  # radians


Note that this is a small positive angle consistent with the sketch.  When \theta > \theta', the pressure in region 2 is higher than in region 2', and the stream must deflect from high pressure to low.  In degrees, the flow angle is just 2.47 degrees.

In [4]:
180*ang/np.pi


2.4762922817144464

Evaluate the states with the now known angle:

In [5]:
s3 = lower(ang)
s3prime = upper(ang)


Their (equal) pressures are:

In [6]:
s3.pres
s3prime.pres


0.06580036555598157

Their Mach numbers are

In [7]:
s3.mach
s3prime.mach


2.400438804183961

Regular reflection of shocks of the same family.  A cananocial case is shown in the sketch, and this configuration is possible when the angles \theta_1 and \theta_2 are sufficiently small.  

For given values of \theta_1, \theta_2, and M_1, the solution, if it exists, is given by matching the flow direction and pressure between regions 2' and 4.  Depending on the parameter values, solutions exist where the wave reflected from the triple point is either an oblique shock or a PM expansion.  This reflected wave is always quite weak.

Example: Take M_1=3, \theta_1 = \theta_2 = 10^0.  Detemine the states 4 and 2'.

In [8]:
upper = lambda ang: GasState.init(3, 1.4).deflect(ang, fam=+1)
lower = lambda ang: GasState.init(3, 1.4).deflect(+10*np.pi/180, fam=+1).deflect(+10*np.pi/180, fam=+1).deflect(ang-20*np.pi/180, fam=-1)


Note that the angle argument in deflect is relative to the input state, so we need to subtract 20 degrees in the last call to deflect for the lower stream to match it with the upper one.  Now match the pressures in 2 and 4':

In [9]:
zerofun = lambda ang: upper(ang).pres - lower(ang).pres
ang = root_scalar(zerofun, bracket=[15 * np.pi / 180, 25 * np.pi / 180], method="brentq").root  # radians
180 * ang / np.pi


20.14266321230273

Note the angle in 2' and 4 is just slightly greater than 20 degrees, so the reflected wave is a very weak oblique shock in this case.  Evaluate the states with the now known angle:

In [10]:
s4 = lower(ang)
s2prime = upper(ang)


Interesting to look at this on a presssure deflection diagram

In [11]:
# Plot helper not yet implemented in the Python port:
# fig1 = presdef(s4)


Note that state 4 must lie on the red pressure-deflection curve for state 1 since the oblique shock from 1 to 2' matches the pressure and deflection angle in state 4.  Note that state 3 (blue curve) is very close to state 4 (cyan curve) because the PM expansion from 3 to 4 is very weak.

Regular of an oblique shock at a free surface.  A cananocial case is shown in the sketch.  This interaction is possible for a oblique shock provided the post-schok flow is supersonic.

As an example, take M_1=4 and an incicdent shock angle \beta = 50^0.  We want to find the angle in flow region 3.  The first difficulty is that we do not know the deflection angle of the flow, so we cannot use 'deflect' directly.  It is also difficult to use a function handle in this case, because the shock angle 'bet' is the 2nd (optional) output of 'deflect'.  I solve this by defining a function 'shockang' which you can find at the bottom of this script (functions in live scripts must be at the end).

In [12]:
def shockang(th):
    beta, _, _ = oblique(4, th, 1.4)
    return beta

zerofun = lambda th: np.pi * 50 / 180 - shockang(th)
th = root_scalar(zerofun, x0=np.pi * 20 / 180, x1=np.pi * 21 / 180, method="secant").root
180 * th / np.pi


np.float64(33.06979278350267)

Now we need to do a PM expansion such that the pressure downstream (region 3) is the same as the original pressure (region 1):

In [13]:
zerofun = lambda ang: GasState.init(4, 1.4).pres - GasState.init(4, 1.4).deflect(th, fam=+1).deflect(ang, fam=-1).pres
ang = root_scalar(zerofun, x0=np.pi * 30 / 180, x1=np.pi * 31 / 180, method="secant").root
GasState.init(4, 1.4).deflect(th, fam=+1).deflect(ang, fam=-1)


GasState(m=3.1681434517079143, gamma=1.4, p=None, T=None, mw=None, p0=0.31067771681232403, t0=1.0, angle=np.float64(1.2311819351273021), proc='pm expansion', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=GasState(m=1.6191355805523613, gamma=1.4, p=None, T=None, mw=None, p0=0.31067771681232403, t0=1.0, angle=np.float64(0.5771767670243819), proc='oblique shock (50.00 deg)', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=GasState(m=4, gamma=1.4, p=None, T=None, mw=None, p0=1.0, t0=1.0, angle=0.0, proc='initstate', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=None, defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k'))), defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k'))), defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k')))

Interestingly, the flow turns by over 70 degrees and comes back up to M=3.2.

Here is the function we needed for the above example

In [14]:
def shockang(th):
    beta, _, _ = oblique(4, th, 1.4)
    return beta
